### Import

In [ ]:
import pandas as pd
import numpy as np
import os
import joblib

# Scikit-learn
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Visualisation
import plotly.graph_objects as go

### Chargement des données et split

In [ ]:
df = pd.read_csv('../data/insurance.csv')
df = df.drop_duplicates().reset_index(drop=True)

numerical_features = ['age', 'bmi', 'children']
categorical_features = ['sex', 'smoker', 'region']
target = 'charges'

X = df[numerical_features + categorical_features]
y = df[target]
# Split avant encodage evite le data leakage
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
def get_feature_names_out(pipeline):
    """Extrait les noms des features après preprocessing + PolynomialFeatures"""
    preprocessor = pipeline.named_steps['preprocessor']
    
    # Noms numériques
    num_features = numerical_features
    
    # Noms catégoriels : ici, 'cat' est directement un OneHotEncoder
    cat_encoder = preprocessor.named_transformers_['cat']  # <-- pas de .named_steps !
    cat_feature_names = cat_encoder.get_feature_names_out(categorical_features)
    
    # Combiner
    feature_names = list(num_features) + list(cat_feature_names)
    
    # Appliquer PolynomialFeatures
    poly = pipeline.named_steps['interactions']
    feature_names_poly = poly.get_feature_names_out(feature_names)
    return feature_names_poly

### Préprocesseur

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_features)
    ],
    remainder='drop'
)

### Fonction de pipeline avec interactions

In [ ]:
def create_pipeline_with_interactions(model):
    return Pipeline([
        ('preprocessor', preprocessor),
        ('interactions', PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)),
        ('regressor', model)
    ])

### Définition des modèles

In [ ]:
pipelines = {
    'Linear': create_pipeline_with_interactions(LinearRegression()),
    'Ridge': create_pipeline_with_interactions(Ridge()),
    'Lasso': create_pipeline_with_interactions(Lasso(max_iter=5000, tol=1e-3))
}

model_names = list(pipelines.keys())
tunable_models = ['Ridge', 'Lasso']

### Recherche large (RandomizedSearchCV)

In [ ]:
print("🔍 Étape 1 : Recherche large avec RandomizedSearchCV")
wide_alpha_ridge = np.geomspace(1e-2, 1e4, num=50)   # [0.01, 10000]
wide_alpha_lasso = np.geomspace(1e-4, 1e2, num=50)   # [0.0001, 100]

wide_param_dists = {
    'Ridge': {'regressor__alpha': wide_alpha_ridge},
    'Lasso': {'regressor__alpha': wide_alpha_lasso}
}

coarse_best_params = {}

for name in tunable_models:
    print(f" → Randomized search for {name}...")
    random_search = RandomizedSearchCV(
        pipelines[name],
        param_distributions=wide_param_dists[name],
        n_iter=50,
        cv=5,
        scoring='neg_mean_squared_error',
        n_jobs=-1,
        random_state=42
    )
    random_search.fit(X_train, y_train)
    coarse_best_params[name] = random_search.best_params_['regressor__alpha']
    print(f"   Meilleur alpha approximatif : {coarse_best_params[name]:.4f}")


### Recherche fine (GridSearchCV)

In [ ]:
print("\n Étape 2 : Recherche fine avec GridSearchCV")
refined_param_grids = {
    'Linear': {},
    'Ridge': {'regressor__alpha': np.geomspace(coarse_best_params['Ridge']/10, coarse_best_params['Ridge']*10, num=25)},
    'Lasso': {'regressor__alpha': np.geomspace(coarse_best_params['Lasso']/10, coarse_best_params['Lasso']*10, num=25)}
}

best_models = {}
cv_results = {}

for name in model_names:
    grid = GridSearchCV(
        pipelines[name],
        refined_param_grids[name],
        cv=5,
        scoring='neg_mean_squared_error',
        n_jobs=-1
    )
    grid.fit(X_train, y_train)
    
    best_models[name] = grid.best_estimator_
    cv_results[name] = {
        'best_alpha': grid.best_params_.get('regressor__alpha', 'N/A'),
        'best_cv_score (RMSE)': np.sqrt(-grid.best_score_)
    }
    
    # Sauvegarde joblib
    os.makedirs('models', exist_ok=True)
    joblib.dump(grid.best_estimator_, f"models/{name.lower()}_model.joblib")
    print(f"   Modèle sauvegardé avec alpha = {grid.best_params_.get('regressor__alpha', 'N/A')}")


In [ ]:
def afficher_coefficients_modele(pipeline, model_name, numerical_features, categorical_features):
    """
    Affiche les coefficients d'un modèle entraîné dans un pipeline,
    avec le style clair de l'exemple fourni.
    """
    # récupérer les vrais noms de features
    preprocessor = pipeline.named_steps['preprocessor']
    num_features = numerical_features
    cat_encoder = preprocessor.named_transformers_['cat']
    cat_feature_names = cat_encoder.get_feature_names_out(categorical_features)
    feature_names_base = list(num_features) + list(cat_feature_names)
    # récupérer le nom des features après interactions créer par PolynomialFeatures
    poly = pipeline.named_steps['interactions']
    feature_names = poly.get_feature_names_out(feature_names_base).tolist()
    
    # récupérer coefficients et intercept
    regressor = pipeline.named_steps['regressor']
    coef = regressor.coef_
    intercept = regressor.intercept_
    
    # affichage
    print(f"\n{'='*70}")
    print(f"COEFFICIENTS DU MODÈLE {model_name.upper()}")
    print(f"{'='*70}")
    print(f"\nIntercept (β₀) : {intercept:,.2f} €")
    print("\nCoefficients des features :")
    
    # Pour Lasso, on filtre les coeffs ≈ 0
    if model_name == 'Lasso':
        indices = [i for i, c in enumerate(coef) if abs(c) > 1e-6]
        print(f"→ Seulement {len(indices)} features non nulles :\n")
    else:
        indices = range(len(coef))
    
    # Trier par importance absolue
    indices = sorted(indices, key=lambda i: abs(coef[i]), reverse=True)
    
    for i in indices[:20]:  # limite à 20 pour lisibilité
        feat = feature_names[i].replace(' ', '_')
        c = coef[i]
        # Tronquer les noms très longs
        display_feat = feat[:30]
        print(f"  β_{display_feat:30} : {c:12,.2f} €")
    
    # interprétation métier
    print(f"\n[Interprétation comme dans ton exemple]")
    try:
        # Effet du tabac
        if 'smoker_yes' in feature_names:
            idx = feature_names.index('smoker_yes')
            print(f"   • Être fumeur → +{coef[idx]:,.0f} €")
        
        # Variables numériques de base
        for feat_name in numerical_features:
            if feat_name in feature_names:
                idx = feature_names.index(feat_name)
                effet = "→ +" if coef[idx] >= 0 else "→ "
                print(f"   • +1 {feat_name} {effet}{abs(coef[idx]):,.0f} €")
                
    except ValueError:
        print("   • Certaines features de base ne sont pas présentes")

In [ ]:
# Affichage des coefficients pour les 3 modèles

for name in ['Linear', 'Ridge', 'Lasso']:
    afficher_coefficients_modele(
        pipeline=best_models[name],
        model_name=name,
        numerical_features=numerical_features,
        categorical_features=categorical_features
    )

### Évaluation sur jeu de test

In [ ]:
print("\n Résultats finaux sur le jeu de test :")
final_results = {}

for name, model in best_models.items():
    y_pred = model.predict(X_test)
    final_results[name] = {
        'MAE': mean_absolute_error(y_test, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_test, y_pred)),
        'R²': r2_score(y_test, y_pred),
        'best_alpha': cv_results[name]['best_alpha']
    }
    alpha_str = final_results[name]['best_alpha']
    if isinstance(alpha_str, float):
        alpha_str = f"{alpha_str:.4f}"
    print(f"\n{name} (α={alpha_str}):")
    print(f"  MAE : {final_results[name]['MAE']:.2f}")
    print(f"  RMSE : {final_results[name]['RMSE']:.2f}")
    print(f"  R² : {final_results[name]['R²']:.4f}")


### Tableau comparatif

In [ ]:
data = []
for name, metrics in final_results.items():
    if name == 'Linear':
        strengths = "Simple, interprétable, pas d'hyperparamètre"
        weaknesses = "Sensible au surapprentissage si features corrélées"
    elif name == 'Ridge':
        strengths = "Stable, gère la multicolinéarité, garde toutes les features"
        weaknesses = "Moins interprétable que Linear, nécessite tuning d'alpha"
    else:  # Lasso
        strengths = "Sélectionne les features utiles, sparse"
        weaknesses = "Peut éliminer des features pertinentes, instable si features corrélées"
    
    data.append({
        'Modèle': name,
        'MAE ($)': round(metrics['MAE'], 2),
        'RMSE ($)': round(metrics['RMSE'], 2),
        'R²': round(metrics['R²'], 4),
        'Alpha': metrics['best_alpha'] if metrics['best_alpha'] != 'N/A' else 'N/A',
        'Points forts': strengths,
        'Points faibles': weaknesses
    })

df_summary = pd.DataFrame(data)
print("\n", df_summary.to_string(index=False))

### Visualisation (tableau Plotly)

In [ ]:
fig = go.Figure(data=[go.Table(
    header=dict(
        values=list(df_summary.columns), 
        fill_color='paleturquoise', 
        align='left', 
        font_size=12, 
        height=30),
    cells=dict(
        values=[df_summary[col] for col in df_summary.columns], 
        fill_color='lavender', 
        align='left', 
        font_size=11, 
        height=25)
)])

fig.update_layout(title="Comparatif des modèles de régression", title_x=0.5, width=1000, height=400)
fig.show()

In [ ]:

# Visualisation : Prédiction vs Réalité (Lasso, Ridge, Linear)

from plotly.subplots import make_subplots


fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=[f"{name} (R² = {final_results[name]['R²']:.3f})" for name in model_names],
    shared_xaxes=True,
    shared_yaxes=True,
    horizontal_spacing=0.05
)

# Couleurs distinctes
colors = {'Linear': 'lightseagreen', 'Ridge': 'mediumpurple', 'Lasso': 'salmon'}

for i, name in enumerate(model_names):
    y_pred = best_models[name].predict(X_test)
    
    fig.add_trace(
        go.Scatter(
            x=y_test,
            y=y_pred,
            mode='markers',
            marker=dict(color=colors[name], opacity=0.6, size=4),
            name=name,
            showlegend=False
        ),
        row=1, col=i+1
    )
    
    # Ligne de parfaite prédiction
    fig.add_trace(
        go.Scatter(
            x=[y_test.min(), y_test.max()],
            y=[y_test.min(), y_test.max()],
            mode='lines',
            line=dict(color='black', dash='dash', width=1),
            showlegend=False
        ),
        row=1, col=i+1
    )

# Mise en page
fig.update_layout(
    title="Comparaison des modèles : Prédiction vs Réalité",
    title_x=0.5,
    width=1200,
    height=400,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5)
)

# Labels communs
fig.update_xaxes(title_text="Vraies valeurs (charges)", row=1, col=2)
fig.update_yaxes(title_text="Prédictions", row=1, col=1)

fig.show()

In [ ]:
# Comparaison prédiction vs réalité pour le modèle sélectionné (ex: Lasso)
selected_model_name = 'Lasso'
y_pred_selected = best_models[selected_model_name].predict(X_test)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=y_test,
    y=y_pred_selected,
    mode='markers',
    marker=dict(color='lightcoral', opacity=0.6),
    name='Prédictions'
))
fig.add_trace(go.Scatter(
    x=[y_test.min(), y_test.max()],
    y=[y_test.min(), y_test.max()],
    mode='lines',
    line=dict(color='royalblue', dash='dash'),
    name='Parfaite prédiction (y = x)'
))

fig.update_layout(
    title=f"Prédiction vs Réalité — Modèle {selected_model_name}",
    xaxis_title="Vraies valeurs (charges)",
    yaxis_title="Prédictions",
    width=700,
    height=600
)
fig.show()

In [ ]:
def predict_insurance_charges_v2(
    model_name="ridge",
    age=30,
    bmi=25.0,
    children=0,
    sex="female",
    smoker=False,
    region="southwest"
):
    """
    Prédit les frais médicaux à partir des données BRUTES (comme dans le CSV original).
    """

#Charger le modèle,
    pipe = joblib.load(f"models/{model_name.lower()}_model.joblib")

#Créer un DataFrame avec les données BRUTES (comme dans df original),
    client = pd.DataFrame([{
        'age': age,
        'bmi': bmi,
        'children': children,
        'sex': sex,
        'smoker': 'yes' if smoker else 'no',
        'region': region
    }])

#Prédire,
    prediction = pipe.predict(client)[0]

#Optionnel : éviter les négatifs (si tu n'as pas fait log(y)),
    if prediction < 0:
        print(" Attention : prédiction négative ! Envisager log(charges).")

    return prediction

In [ ]:
client_args = dict(age=30, bmi=24.0, children=1, sex="female", smoker=False)

for model in ["linear", "ridge", "lasso"]:
    charge = predict_insurance_charges_v2(model_name=model, **client_args)
    print(f"{model.capitalize():>8} : ${charge:,.2f}")

## Conclusion

Le modèle Lasso constitue une base solide, transparente et adaptée à la nature de nos données : peu nombreuses, propres, et explicites. Il répond aux exigences de performance tout en offrant une lisibilité précieuse pour les métiers de l’assurance. La prochaine étape consisterait à combiner cette rigueur linéaire avec la flexibilité des modèles modernes, afin de capturer d’éventuelles non-linéarités sans sacrifier l’explicabilité — un équilibre essentiel pour une tarification juste et compréhensible.